## Configuracion

In [1]:
!pip install selenium
!pip install beautifulsoup4
!pip install html5lib
!pip install pandas openpyxl


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import pandas as pd
import json
import re

#Configuraciones iniciales para el uso en colab
# Configurar Chrome para headless mode
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# Specify the path to your ChromeDriver executable (if not in PATH)
# chrome_driver_path = '/path/to/chromedriver'  # Uncomment and replace with the actual path

# Inicializar el driver de Chrome
# If ChromeDriver is in your PATH, you can use the following line:
driver = webdriver.Chrome(options=chrome_options)
# If ChromeDriver is not in your PATH, use the following line:
# driver = webdriver.Chrome(options=chrome_options, executable_path=chrome_driver_path)


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import pandas as pd
import json
import re

In [ ]:
#Configuraciones iniciales para el uso en colab
# Configurar Chrome para headless mode
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# Inicializar el driver de Chrome
driver = webdriver.Chrome(options=chrome_options)


## URL DE EXTRACCION


In [ ]:
#base_url = "https://www.carrefour.com.ar/Electro-y-tecnologia"  <#tiene json
#base_url = "https://www.vea.com.ar/electro" #tiene json
##base_url ="https://supermercado.laanonimaonline.com/almacen/n1_1/"   #No cuenta con busqueda por ean (es por id) .tiene script
#base_url ="https://www.disco.com.ar/electro" #distintos nombres de campos, si tiene .tiene json
#base_url ="https://www.masonline.com.ar/electroytecno" #   <3tiene json



```
# Tiene formato de código
```

# Extracion de json:
      En este caso estamos buscado un scrip que diga  {'type': 'application/ld+json'})

In [ ]:


# Función para extraer el contenido del script en JSON-LD de una página
def extraer_json_ld(html):
    soup = BeautifulSoup(html, 'html.parser')
    div_etiqueta = soup.find('div', {'class': 'flex flex-column min-vh-100 w-100'})

    # Si se encuentra el div, buscamos el script con type="application/ld+json"
    if div_etiqueta:
        script_tag = div_etiqueta.find('script', {'type': 'application/ld+json'})
        if script_tag:
            return script_tag.string
    return None


### Itercion en hojas


In [ ]:
# Inicializar lista para almacenar los datos extraídos
data = []
# Número máximo de páginas que se desea iterar
max_pages = 10

In [ ]:
# Iterar sobre el número de páginas
for page in range(1, max_pages + 1):
    # Construir la URL de la página
    url = base_url.format(page)

    # Cargar la página
    driver.get(url)

    # Esperar unos segundos para que cargue completamente (opcional)
    time.sleep(3)

    # Obtener el HTML completo de la página cargada
    html = driver.page_source

    # Extraer el contenido JSON-LD de la página actual
    json_ld_content = extraer_json_ld(html)

    if json_ld_content:
        # Agregar los datos a la lista
        data.append({
            'page': page,
            'json_ld': json_ld_content
        })

    # Imprimir para verificar el progreso
    print(f"Página {page} procesada.")

# Cerrar el navegador después de la iteración
driver.quit()

Página 1 procesada.
Página 2 procesada.
Página 3 procesada.
Página 4 procesada.
Página 5 procesada.
Página 6 procesada.
Página 7 procesada.
Página 8 procesada.
Página 9 procesada.
Página 10 procesada.


In [ ]:
# Convertir los datos procesados a un DataFrame
df_final = pd.DataFrame(data)

# Guardar el DataFrame en un nuevo archivo Excel
df_final.to_excel('mi_lista.xlsx', index=False)

print("Datos procesados y guardados exitosamente en productos_procesados.xlsx")

Datos procesados y guardados exitosamente en productos_procesados.xlsx


# Lectura de json

In [ ]:
import pandas as pd
import json
import re

# Definir la expresión regular para extraer el EAN de la URL
ean_regex = re.compile(r'/(\d+)_')  # Ajusta esta expresión según el formato específico de la URL

# Leer el archivo Excel
df = pd.read_excel('/content/mi_lista.xlsx')

# Lista para almacenar todos los productos
all_products = []

# Iterar sobre cada fila del DataFrame
for _, row in df.iterrows():
    # Cargar el JSON
    json_data = json.loads(row['json_ld'])

    # Extraer la lista de productos
    items = json_data.get('itemListElement', [])

    # Añadir los productos a la lista
    for item in items:
        producto = item.get('item', {})

        # Extraer la URL de la imagen
        image_url = producto.get('image', '')

        # Intentar extraer el EAN usando la expresión regular
        ean_match = ean_regex.search(image_url)
        ean = ean_match.group(1) if ean_match else None

        # Crear un diccionario con los campos requeridos
        son_item = {
            "Nombre del producto": producto.get('name'),
            "Marca": producto.get('brand', {}).get('name'),
            "Precio más bajo": producto.get('offers', {}).get('lowPrice'),
            "Precio más alto": producto.get('offers', {}).get('highPrice'),
            "Moneda": producto.get('offers', {}).get('priceCurrency'),
            "Disponibilidad": producto.get('offers', {}).get('offers', [])[0].get('availability') if producto.get('offers', {}).get('offers') else None,
            "Condición": producto.get('offers', {}).get('offers', [])[0].get('itemCondition') if producto.get('offers', {}).get('offers') else None,
            "Válido hasta": producto.get('offers', {}).get('offers', [])[0].get('priceValidUntil') if producto.get('offers', {}).get('offers') else None,
            "URL del producto": producto.get('@id'),
            "EAN": ean  # Añadir el EAN a los datos
        }

        all_products.append(son_item)

# Crear un DataFrame con todos los productos
df_products = pd.DataFrame(all_products)

# Convertir a archivo Excel
df_products.to_excel('/content/productos.xlsx', index=False)


JSONDecodeError: Unterminated string starting at: line 1 column 31846 (char 31845)